# Integridad · Identidad y partidas

[Inicio](../../tests.ipynb) · Anterior: [HU-06 y HU-07 · Salas](../funcional/hu06_hu07_salas.ipynb) · Siguiente: [Integridad · Salas y borrados](salas_y_borrados.ipynb)

Las fichas incluyen los criterios del Plan Maestro. Las referencias U/G/F/W/D y las rutas abreviadas en **Evidencia** se interpretan en la [matriz maestra](../../docs/test/qa-master-matrix.md). Una ejecución local no implica un resultado publicado ni prueba un control de entrega.

## RI-01

- **Criterio:** El correo y el nombre de usuario son únicos; un registro duplicado responde 409, nunca 500.
- **Método previsto:** API, WebSocket o SQL, según la capa. Garantía prevista: BD: UNIQUE + rollback.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**P**): `features/registration.feature` (**G**) ejercita colisiones simultáneas en PostgreSQL por correo, usuario y ambos: 201/409, una sola fila y alta posterior válida. `test_auth_service.py` es una referencia histórica ausente; el escenario vigente `Email conflict takes precedence even when two accounts own the fields` comprueba el 409 secuencial del conflicto doble, no todos los casos secuenciales por separado.
  
  Evidencia local posterior o adicional: En PostgreSQL desechable se verificaron 20 colisiones simultáneas consecutivas de correo (12,592 s), cada una con 201/409, una fila, bcrypt y registro posterior válido (Engram #932). La matriz registra también choques por usuario y por ambos. [Fuente de prueba](../../backend/tests/features/registration.feature).
- **Brecha o límite:** Las 20 repeticiones exigidas en §5.2 son de correo; no se atribuyen a usuario o choque doble.

**Procedimiento del Plan (§5.2)**

TC-INTEG-01: colisión concurrente de correo y comprobación de RI-02.

- **Preparación:** PostgreSQL de prueba sin `race@typr.test` y con UNIQUE activo para correo y nombre.
- **Pasos:** Enviar en dos hilos dos POST a `/api/v1/auth/register` con ese correo y usuarios `race a` y `race b`; revisar ambos HTTP, contar filas y revisar la contraseña guardada.
- **Éxito:** 201 y 409 con rollback, nunca 500; exactamente una fila y hash bcrypt. Repetir 20 veces consecutivas sin duplicados ni errores 500.

<details>
<summary>Texto literal de la ficha del Plan</summary>

HU-01 · services/auth.py · tabla de usuarios (PostgreSQL) Integridad de datos y concurrencia Crítica (P1) / Bloqueante Automatizada (unittest con hilos concurrentes en docker-compose.test.yml) 1. Base de datos de prueba sin el correo race@typr.test. 2. Restricción UNIQUE activa en correo y nombre de usuario (init.sql). 1. Enviar al mismo tiempo dos POST /api/v1/auth/register con el mismo correo y distinto usuario. 2. Revisar los códigos HTTP de ambas respuestas. 3. Contar las filas de usuarios con ese correo. 4. Revisar la contraseña guardada del registro creado. Correo race@typr.test · usuarios race a y race b · 2 hilos simultáneos _ _ 1. Una petición responde 201 Created. 2. La otra responde 409 Conflict (con rollback), nunca 500. 3. Existe exactamente 1 fila con ese correo. 4. La contraseña está guardada como hash bcrypt (regla RI-02). Sin usuarios duplicados ni errores 500 en 20 repeticiones consecutivas.

</details>

**Gherkin ejecutable (original en inglés)** · [Fuente](../../backend/tests/features/registration.feature).

```gherkin
@auth @postgresql
Feature: Registration preserves unique account identities and protects passwords

  @RI-01
  Scenario Outline: Overlapping registrations collide on <identity>
    Given two new registrations that share <identity>
    When both registrations reach the database insert in separate transactions
    Then one request succeeds and the other receives "<detail>" with status 409
    And exactly one matching account is persisted
    And a new unrelated account can still register

    Examples:
      | identity | detail                      |
      | email    | Este correo ya está en uso  |
      | username | El usuario ya existe bro    |
      | both     | Este correo ya está en uso  |

  @RI-01
  Scenario: Email conflict takes precedence even when two accounts own the fields
    Given two registered accounts with different emails and usernames
    When a new request uses the email of one and the username of the other
    Then registration returns "Este correo ya está en uso" with status 409
```

## RI-02

- **Criterio:** La contraseña se guarda solo como hash bcrypt y ninguna respuesta de la API incluye el hash.
- **Método previsto:** API, WebSocket o SQL, según la capa. Garantía prevista: Servidor.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**V**): `features/registration.feature` (**G**) verifica hash bcrypt persistido y ausencia de contraseña/hash en registro, login y perfil. [Fuente de prueba](../../backend/tests/features/registration.feature).
- **Brecha o límite:** No se identificó una brecha para el escenario y entorno documentados.

**Gherkin ejecutable (original en inglés)** · [Fuente](../../backend/tests/features/registration.feature).

```gherkin
@auth @postgresql
Feature: Registration preserves unique account identities and protects passwords

  @TC-FUNC-01 @RI-02
  Scenario: Passwords are stored as bcrypt and never exposed by account endpoints
    When a new account registers, logs in, and requests its profile
    Then its persisted password is a bcrypt hash of the submitted password
    And the register, login, and profile responses omit password material
```

## RI-03

- **Criterio:** Toda partida se registra bajo el usuario del token; el cliente no puede indicar otro usuario.
- **Método previsto:** API, WebSocket o SQL, según la capa. Garantía prevista: Servidor (JWT).
- **Estado:** **V — Verificado**.
- **Evidencia:** El corte histórico (**P**) atribuye titular e historial privado a `backend/tests/features/practice.feature`, ausente del árbol actual. El Gherkin vigente de participaciones comprueba el rechazo de identidad ajena y el titular del token en práctica; la consulta del historial se comprueba aparte en [TC-FUNC-05](../funcional/hu02_practica.ipynb#TC-FUNC-05).
  
  Evidencia local posterior o adicional: La API rechazó `userId` ajeno con 422, sin crear partida; una petición válida se guardó bajo el titular del token. PostgreSQL 16/Alembic 0005: 2 escenarios y 9 pasos compartidos con RI-04 y RI-05 (Engram #993). [Fuente de prueba](../../backend/tests/features/participation.feature).
- **Brecha o límite:** Alcance: envío explícito de identidad ajena en práctica.

**Gherkin ejecutable (original en inglés)** · [Fuente](../../backend/tests/features/participation.feature).

```gherkin
Feature: Completed matches preserve participation integrity

  @RI-03 @RI-04
  Scenario: Practice is recorded only for the signed-in player and without a room
    Given two signed-in practice players
    When the first player submits practice with the second player's user ID
    Then the client-supplied owner is rejected without recording a match
    When the first player submits the same practice without an owner field
    Then one solo participation exists for the signed-in player and has no room
```

La referencia histórica a `practice.feature` no está disponible; se reproduce la prueba existente de participaciones.

## RI-04

- **Criterio:** Partida individual: una participación, modo individual y sin sala. Multijugador: una participación por jugador y con sala.
- **Método previsto:** API, WebSocket o SQL, según la capa. Garantía prevista: Servidor + BD (FK).
- **Estado:** **V — Verificado**.
- **Evidencia:** El corte histórico (**V**) cita `backend/tests/features/practice.feature` para la modalidad individual; ese archivo no está disponible. Los dos escenarios vigentes de `participation.feature` (`@RI-04`) verifican ambas modalidades; `test_ws_rooms.py` complementa el vínculo a sala y los resultados de ambos jugadores.
  
  Evidencia local posterior o adicional: La corrida de RI-03 y RI-05 persistió práctica con un participante y sin sala, y carrera con ambos jugadores y sala activa (Engram #993). [Fuente de prueba](../../backend/tests/features/participation.feature).
- **Brecha o límite:** El vínculo a la sala solo se mantiene mientras exista; al borrarla se aplica `SET NULL`.

**Gherkin ejecutable (original en inglés)** · [Fuente](../../backend/tests/features/participation.feature).

```gherkin
Feature: Completed matches preserve participation integrity

  @RI-03 @RI-04
  Scenario: Practice is recorded only for the signed-in player and without a room
    Given two signed-in practice players
    When the first player submits practice with the second player's user ID
    Then the client-supplied owner is rejected without recording a match
    When the first player submits the same practice without an owner field
    Then one solo participation exists for the signed-in player and has no room

  @RI-04 @RI-05
  Scenario: A completed race stores exactly one participation per player
    Given a live race with two signed-in members
    When both members complete the race
    Then the game is linked to the live room and has one participation per player
    And PostgreSQL rejects a duplicate game and user pair without adding a row
```

La referencia histórica a `practice.feature` no está disponible; se reproducen los dos escenarios de participaciones existentes.

## Reproducción local opcional

Prepare el entorno desechable de la [guía de pruebas](../../test/README.md). Ejecute solo la celda que necesita, con el kernel iniciado en la raíz de Typr o dentro de `notebooks/`. No ejecute en paralelo. Abrir el cuaderno no inicia pruebas; la salida describe una corrida local, no cambia el estado de las fichas. Las celdas de backend ejecutan la suite indicada completa, no solo los casos de esta página.

### Criterios de aceptación · RI-01 a RI-04

**Plan (§§5.2 y 5.6):** El correo y el usuario son únicos; un duplicado recibe 409, nunca 500. La contraseña solo se guarda como bcrypt y el hash no sale por la API. El resultado se atribuye al usuario del token; el modo individual tiene una participación sin sala y el multijugador, una por jugador con sala.

**Qué comprueba esta celda:** Ejecuta todo el Gherkin del backend, sin filtro. `registration.feature` comprueba colisiones de correo, usuario y ambos: una alta, un 409, una sola fila y una alta posterior posible. Comprueba bcrypt y ausencia de material de contraseña en registro, login y perfil. `participation.feature` comprueba el rechazo de un propietario ajeno sin guardar partida, y las participaciones individual y multijugador con sus vínculos esperados.

**Límite:** No repite automáticamente las 20 colisiones de correo del plan. El escenario multijugador se comparte con RI-05.

In [ ]:
from pathlib import Path
from runpy import run_path

directorio = Path.cwd().resolve()
for raiz in (directorio, *directorio.parents):
    if (raiz / "docker-compose.test.yml").is_file() and (raiz / "notebooks/ejecutar_qa.py").is_file():
        break
else:
    raise RuntimeError("Inicie el kernel desde la raíz de Typr o dentro de notebooks/.")

run_path(str(raiz / "notebooks/ejecutar_qa.py"))["ejecutar"]('backend-gherkin')

[Inicio](../../tests.ipynb) · Anterior: [HU-06 y HU-07 · Salas](../funcional/hu06_hu07_salas.ipynb) · Siguiente: [Integridad · Salas y borrados](salas_y_borrados.ipynb)